# 4.20 Resumen y Consolidación de Resultados de Kaggle

Este notebook consolida los resultados de las corridas del experimento con los puntajes obtenidos en el leaderboard público de Kaggle.

### Objetivo:
1. Utiliza la variable `experimento <- "exp420_00"`.
2. Busca dentro del directorio del experimento los archivos `resumen_corridas.txt` y `resumen_kaggle.txt`.
3. Para los nombres de archivos que se encuentren en ambos archivos, extrae y combina los hiperparámetros, resultados y el score de Kaggle.
4. Agrega los registros junto con el identificador del experimento al archivo `datasets/corridas-arboles-azarosos.csv`.

#### 1. Limpieza del Ambiente y Carga de Librerías

In [1]:
# Limpieza de memoria y ambiente
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

# Carga de librerías necesarias
require("data.table")

format(Sys.time(), "%a %b %d %X %Y")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,657948,35.2,1454614,77.7,1114793,59.6
Vcells,1223445,9.4,8388608,64.0,1975054,15.1


Loading required package: data.table



[1] "Mon Aug 24 14:50:21 2026"

#### 2. Configuración de Experimento y Rutas de Archivos

In [2]:
# Variable de experimento
experimento <- "exp420_04"

# Función para determinar la raíz del proyecto
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()

# Rutas a los directorios y archivos
dir_exp <- file.path(dir_base, "exp", experimento)
path_corridas <- file.path(dir_exp, "resumen_corridas.txt")
path_kaggle <- file.path(dir_exp, "resumen_kaggle.txt")
path_destino <- file.path(dir_base, "datasets", "corridas-arboles-azarosos.csv")

cat("Directorio base:        ", dir_base, "\n")
cat("Directorio experimento: ", dir_exp, "\n")
cat("Archivo corridas:       ", path_corridas, "\n")
cat("Archivo Kaggle:         ", path_kaggle, "\n")
cat("Archivo destino CSV:    ", path_destino, "\n")

Directorio base:         /workspace 
Directorio experimento:  /workspace/exp/exp420_04 
Archivo corridas:        /workspace/exp/exp420_04/resumen_corridas.txt 
Archivo Kaggle:          /workspace/exp/exp420_04/resumen_kaggle.txt 
Archivo destino CSV:     /workspace/datasets/corridas-arboles-azarosos.csv 


#### 3. Lectura y Parseo de `resumen_corridas.txt` y `resumen_kaggle.txt`

In [3]:
# Validar existencia de archivos
if (!file.exists(path_corridas)) {
  stop(sprintf("No se encontró el archivo: %s", path_corridas))
}
if (!file.exists(path_kaggle)) {
  stop(sprintf("No se encontró el archivo: %s", path_kaggle))
}

# 1. Lectura de resumen_corridas.txt
tb_corridas <- fread(path_corridas)
cat(sprintf("Registros leídos de resumen_corridas.txt: %d\n", nrow(tb_corridas)))

# 2. Lectura y parseo de resumen_kaggle.txt
lineas_kaggle <- trimws(readLines(path_kaggle, warn = FALSE))
lineas_kaggle <- lineas_kaggle[lineas_kaggle != ""]

archivos_k <- character()
scores_k <- integer()

i <- 1
while (i <= length(lineas_kaggle)) {
  linea_actual <- lineas_kaggle[i]
  
  # Si la línea termina en .csv, corresponde a un archivo de predicción
  if (grepl("\\.csv$", linea_actual, ignore.case = TRUE)) {
    arch_nombre <- basename(linea_actual)
    score_val <- NA_integer_
    
    # Buscar el score en las líneas subsiguientes
    for (offset in 1:3) {
      if (i + offset <= length(lineas_kaggle)) {
        cand <- lineas_kaggle[i + offset]
        # Detectar número de score (ej. 334.581 o 334581)
        if (grepl("^[0-9]+(\\.[0-9]+)?$", cand)) {
          val_num <- as.numeric(cand)
          # Si tiene punto decimal (ej 334.581), convertir a entero multiplicando por 1000
          score_val <- as.integer(round(if (grepl("\\.", cand)) val_num * 1000 else val_num))
          i <- i + offset
          break
        }
      }
    }
    
    if (!is.na(score_val)) {
      archivos_k <- c(archivos_k, arch_nombre)
      scores_k <- c(scores_k, score_val)
    }
  }
  i <- i + 1
}

tb_kaggle <- data.table(archivo = archivos_k, score = scores_k)
# Deduplicar por archivo si hubiese más de un envío
tb_kaggle <- unique(tb_kaggle, by = "archivo", fromLast = TRUE)

cat(sprintf("Registros parseados de resumen_kaggle.txt: %d\n", nrow(tb_kaggle)))

Registros leídos de resumen_corridas.txt: 48
Registros parseados de resumen_kaggle.txt: 177


#### 4. Cruce de Información (Archivos Presentes en Ambos)

In [4]:
v_experimento <- experimento

# Inner join por el nombre de archivo
tb_coincidentes <- merge(tb_corridas, tb_kaggle, by = "archivo")

# Asignar columna con el nombre/número del experimento
tb_coincidentes[, experimento := v_experimento]

# Orden estándar de columnas según datasets/corridas-arboles-azarosos.csv
cols_estandar <- c(
  "experimento", "iter", "feature_fraction", "maxdepth", "minsplit",
  "minbucket_fraction", "minbucket", "cp", "num_trees", "archivo",
  "envios_positivos", "score"
)

setcolorder(tb_coincidentes, cols_estandar)
setorder(tb_coincidentes, iter)

cat(sprintf("Archivos encontrados en ambos resúmenes: %d\n", nrow(tb_coincidentes)))
print(head(tb_coincidentes, 5))

Archivos encontrados en ambos resúmenes: 48
   experimento  iter feature_fraction maxdepth minsplit minbucket_fraction
        <char> <int>            <num>    <int>    <int>              <num>
1:   exp420_04     1              0.1       14      100                0.2
2:   exp420_04     2              0.1       14       80                0.2
3:   exp420_04     3              0.1       14       60                0.2
4:   exp420_04     4              0.1       14       40                0.2
5:   exp420_04     5              0.1       16      100                0.2
   minbucket    cp num_trees
       <int> <int>     <int>
1:        20    -1        32
2:        16    -1        32
3:        12    -1        32
4:         8    -1        32
5:        20    -1        32
                                                archivo envios_positivos  score
                                                 <char>            <int>  <int>
1: KA420_ff_0.10_md_14_ms_100_mb_20_cp_-1.0_32trees.csv             

#### 5. Agregar / Consolidar en `datasets/corridas-arboles-azarosos.csv`

In [5]:
# Asegurar que el directorio de datasets exista
dir.create(dirname(path_destino), showWarnings = FALSE, recursive = TRUE)

if (file.exists(path_destino)) {
  tb_existente <- fread(path_destino)
  cat(sprintf("Registros preexistentes en %s: %d\n", basename(path_destino), nrow(tb_existente)))
  
  # Conservar registros de otros experimentos y reemplazar/actualizar el actual
  tb_otros_exp <- tb_existente[experimento != v_experimento]
  tb_consolidado <- rbind(tb_otros_exp, tb_coincidentes, fill = TRUE)
} else {
  cat("Creando nuevo archivo de consolidación.\n")
  tb_consolidado <- tb_coincidentes
}

# Ordenar por experimento e iter
setcolorder(tb_consolidado, cols_estandar)
setorder(tb_consolidado, experimento, iter)

# Guardar archivo CSV consolidado
fwrite(tb_consolidado, file = path_destino, sep = ",")

cat("\n=========================================\n")
cat("CONSOLIDACIÓN COMPLETADA CON ÉXITO\n")
cat(sprintf("Registros agregados de %s: %d\n", v_experimento, nrow(tb_coincidentes)))
cat(sprintf("Total registros consolidados: %d\n", nrow(tb_consolidado)))
cat(sprintf("Archivo actualizado: %s\n", path_destino))
cat("=========================================\n")

Registros preexistentes en corridas-arboles-azarosos.csv: 190

CONSOLIDACIÓN COMPLETADA CON ÉXITO
Registros agregados de exp420_04: 48
Total registros consolidados: 238
Archivo actualizado: /workspace/datasets/corridas-arboles-azarosos.csv


#### 6. Resumen de Experimentos Consolidados

In [6]:
# Resumen agrupado por experimento
resumen_exp <- tb_consolidado[, .(
  cantidad_corridas = .N,
  score_maximo = max(score, na.rm = TRUE),
  score_promedio = round(mean(score, na.rm = TRUE), 1),
  score_minimo = min(score, na.rm = TRUE)
), by = experimento]

print(resumen_exp)

cat("\nTop 5 mejores modelos globales:\n")
print(head(tb_consolidado[order(-score), .(experimento, iter, feature_fraction, maxdepth, minsplit, minbucket, score, archivo)], 5))

format(Sys.time(), "%a %b %d %X %Y")

   experimento cantidad_corridas score_maximo score_promedio score_minimo
        <char>             <int>        <int>          <num>        <int>
1:   exp420_00                60       353581       339756.4       328415
2:   exp420_01                24       357831       331678.7       308498
3:   exp420_02                10       355581       340273.1       323832
4:   exp420_03                96       366081       344318.3       328915
5:   exp420_04                48       368248       346308.8       317748

Top 5 mejores modelos globales:
   experimento  iter feature_fraction maxdepth minsplit minbucket  score
        <char> <int>            <num>    <int>    <int>     <int>  <int>
1:   exp420_04    42             0.18       18       80        16 368248
2:   exp420_03    77             0.15       14      100        20 366081
3:   exp420_04    17             0.15       14      100        20 366081
4:   exp420_04    46             0.18       20       80        16 364581
5:   exp420

[1] "Mon Aug 24 14:50:21 2026"